# 用講的跟 Workflow 對話

語音互動牽涉兩個模組，本文件示範兩者：

- **Action** — `VoiceAnswerAction` 在單次生成中產出兩個頻道：說出口的內容，以及顯示於畫面的內容。
- **Perceive** — `VoiceTextPerceive` 將語音轉為該回合的輸入，並在偵測到使用者開口時中止進行中的回答。

四個章節依序處理四個主題：語音輸出、語音輸入、兩個頻道的併行輸出，以及回答被中斷時的行為。

本文件全程連線真實端點，不使用替身物件；所有輸出皆為模型即時生成與合成的結果。


## 在 Colab 準備環境


In [ ]:
!pip install -q "git+https://github.com/R300-AI/Agentic-SDK.git"

## 端點設定

以下變數指向本文件使用的端點。若使用的是 OpenAI 端點，下一節可整節略過，直接以 `RealtimeTranscription(api_key=..., model=...)` 建立傳輸。


In [ ]:
AZURE_ENDPOINT      = "https://<資源>.cognitiveservices.azure.com"
AZURE_API_KEY       = "<KEY>"
CHAT_BASE_URL       = "https://<資源>/openai/v1/"
CHAT_API_KEY        = "<KEY>"
CHAT_MODEL          = "<部署名稱>"
TRANSCRIBE_MODEL    = "<部署名稱>"
TTS_MODEL           = "<部署名稱>"
REALTIME_API_VERSION = "2025-04-01-preview"
SPEECH_API_VERSION   = "2025-03-01-preview"

## 連線方式不同的端點

SDK 內附的傳輸僅建立 `OpenAI` client，不內建任何其他供應商的整合。端點的連線方式不同時，覆寫單一方法即可；開啟連線之後的行為——session 設定、音訊傳送、事件分派、回合判定、靜音閘門——均由基底類別提供。

`turn_detection` 決定服務如何判定一段話結束：依固定的靜音長度，或由模型判斷該停頓屬於句中換氣或句末。此參數屬於傳輸，不屬於模組；模組的職責僅限於決定哪些音訊送出。


In [ ]:
from openai import AzureOpenAI
from agentic_sdk.audio.realtime import RealtimeTranscription
from agentic_sdk.audio.speech import SpeechOutput


class MyTranscription(RealtimeTranscription):
    """我自己接的端點。SDK 只認 OpenAI，這一家是我接的。"""

    def _open(self):
        return AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=AZURE_API_KEY,
                           api_version=REALTIME_API_VERSION).beta.realtime.connect(
            model=self._model, extra_query={"intent": "transcription"})


class MySpeech(SpeechOutput):
    def _open_stream(self, text):
        return AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=AZURE_API_KEY,
                           api_version=SPEECH_API_VERSION
                           ).audio.speech.with_streaming_response.create(
            model=self._model, voice=self._voice, input=text,
            response_format=self._response_format)

## 一、文字輸入、語音輸出

語音輸出不以語音輸入為前提。本節的 perceive 為一般的 `PassThroughPerceive`，輸入來自鍵盤，回答以語音產出。

兩個頻道的用途不同：`spoken` 為口語內容，承載判斷與理由；`displayed` 為視覺內容，承載條列、型號與數值。兩者互為補充，而非同一份內容的重複。


In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import PassThroughPerceive, VoiceAnswerAction

class Recorded(MySpeech):
    """把合成出來的音訊留下來，這樣 Colab 可以播給你聽。"""
    def __init__(self, **kw):
        self.audio = bytearray()
        self.said = []
        super().__init__(**kw)
    def speak(self, text):
        self.said.append(text)
        for piece in super().speak(text):
            self.audio.extend(piece)
            yield piece

speaking = Recorded(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
workflow = Workflow(workflow_name="打字問、聲音答",
                    perceive=PassThroughPerceive(), action=action)
result = workflow.run("保固多久？我要能貼在說明頁上的版本。")

print("畫面上顯示 :", result.final_message.replace("\n", " / ")[:80])
print("說出口的   :", speaking.said[0][:60])
print("合成音訊   :", len(speaking.audio), "bytes")

## 二、語音輸入

在缺少音訊裝置的環境中，可由同一個合成端點產生問句音訊，再將該音訊送入轉寫傳輸，即可完整示範語音輸入路徑。

本節呈現兩項行為。其一，低於音量門檻的片段不會離開本機：純靜音與語音的計費相同，且服務會將其辨識為未曾說出的字詞。其二，`run()` 不需再次取得使用者的輸入內容；語音抵達的時機取決於使用者，因此模組先行收取，工作流在執行時取用。


In [ ]:
import array, time
from agentic_sdk.modules import VoiceTextPerceive

def as_microphone(text, *, quiet_after=2.5, rate_in=24000, rate_out=16000):
    """用合成語音假裝有人在講話，這樣沒有麥克風也能示範。

    尾巴要補靜音。真的麥克風在人停止說話之後還是持續在收，而服務就是靠
    「聽到靜音」判定一句話結束——講完就把音訊切掉，轉寫永遠不會回來。
    """
    pcm = b"".join(MySpeech(model=TTS_MODEL).speak(text))
    samples = array.array("h"); samples.frombytes(pcm)
    step = rate_in / rate_out
    out = array.array("h", (samples[int(i * step)] for i in range(int(len(samples) / step))))
    out.extend([0] * int(rate_out * quiet_after))
    frame = 1600                       # 十分之一秒
    return [out[i:i + frame].tobytes() for i in range(0, len(out), frame)]

listening = MyTranscription(model=TRANSCRIBE_MODEL, language="zh",
                            turn_detection={"type": "semantic_vad", "eagerness": "low"})
perceive = VoiceTextPerceive(transport=listening, speech_threshold=500, hangover_seconds=1.2)

# 從轉寫回呼旁觀就好。pending_input() 是「取用」——印出來就等於用掉了，
# 待會 run() 會拿不到東西。
heard = []
listening.on_transcript(heard.append)

chunks = as_microphone("保固期是多久？")
for chunk in chunks:
    perceive.hear(chunk)               # 安靜的片段不會離開這台機器
    time.sleep(0.1)                    # 麥克風是即時來的
for _ in range(60):                    # 等服務判定這句話講完
    if perceive.pending_input():
        break
    time.sleep(0.2)

print("麥克風片段數 :", len(chunks))
print("聽到的話     :", heard[0] if heard else "（還沒回來）")

speaking = Recorded(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
talking = Workflow(workflow_name="用講的問、用聲音答", perceive=perceive, action=action)
result = talking.run()                 # 不必再告訴它使用者說了什麼

print("畫面上顯示   :", result.final_message.replace("\n", " / ")[:60])
print("說出口的     :", speaking.said[0][:50])

## 三、語音與文字併行輸出

`spoken` 欄位完成時即送往合成，不等待整段回覆產生完畢。回覆愈長，提前開口的效果愈顯著。本節以時間量測兩者的差距。


In [ ]:
written = []
started = []

class Timed(Recorded):
    """記下「開始說話」是這一輪的第幾秒。"""
    def speak(self, text):
        started.append(time.monotonic() - began)
        return super().speak(text)

speaking = Timed(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
workflow = Workflow(workflow_name="邊說邊顯示",
                    perceive=PassThroughPerceive(), action=action)

began = time.monotonic()
stream = workflow.stream("保固多久？請給我可以貼在說明頁上的完整條列。")
for piece in stream:
    written.append(piece)
finished = time.monotonic() - began

print(f"開始說話 : 第 {started[0]:.1f} 秒")
print(f"整段寫完 : 第 {finished:.1f} 秒（共 {len(''.join(written))} 個字）")
print(f"→ 聲音比整段文字早了 {finished - started[0]:.1f} 秒")

## 四、回答被中斷

中斷的觸發依據為語音活動，而非轉寫內容。語音活動約於使用者開口後 600 毫秒即可偵測，轉寫則需接近四秒；以轉寫為依據將導致系統在使用者說話期間持續輸出。

中斷後，對話記錄保留的是實際送達使用者的內容，而非模型完整產出的內容。未送達的部分不進入下一回合的上下文，以避免 Agent 引用使用者未曾接收的內容。

`aborted` 在此情境下為 `False`。該旗標用於流程自我中止（跳轉上限、逾時等），並以錯誤形式呈現；使用者主動插話不屬於錯誤。


In [ ]:
from agentic_sdk.core.cancellation import CancellationToken

listening = MyTranscription(model=TRANSCRIBE_MODEL, language="zh",
                            turn_detection={"type": "semantic_vad", "eagerness": "low"})
perceive = VoiceTextPerceive(transport=listening, speech_threshold=500, hangover_seconds=1.2)

class Interrupted(Recorded):
    """一邊播放，一邊讓使用者在中途開口。"""
    def speak(self, text):
        for index, piece in enumerate(super().speak(text)):
            if index == 3:                        # 播到一小段就插話
                for chunk in barge_in:
                    perceive.hear(chunk)
            yield piece

barge_in = as_microphone("等一下，我不是問這個。", quiet_after=1.0)

speaking = Interrupted(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
talking = Workflow(workflow_name="會被打斷的對話", perceive=perceive, action=action)

for chunk in as_microphone("保固期是多久？"):
    perceive.hear(chunk)
    time.sleep(0.1)
for _ in range(60):
    if perceive.pending_input():
        break
    time.sleep(0.2)

cut = talking.run(cancel=CancellationToken())
print("被打斷了嗎 :", cut.interrupted)
print("這是錯誤嗎 :", cut.aborted)
print("對方收到的 :", (cut.interrupt_payload.get("delivered") or "")[:40])
print("記憶裡留的 :", talking.memory.turns[-1].content[:40])

## 音訊裝置的責任歸屬

SDK 不擷取麥克風，也不播放音訊。

輸入側由呼叫端將音訊片段送入 `hear()`；本文件的 `as_microphone` 即為此端的替代實作，實際應用改為音訊裝置的輸出即可。

輸出側需在 `speak()` 外包一層，將每一段音訊交給音訊裝置後再 `yield`。順序具有意義：先播放再 `yield`，中斷時放棄該串流才能同時停止播放與合成。本文件的 `Recorded` 即為此形狀，差別僅在於它保存音訊而非播放。

完整且可執行的範例位於 `examples/voice/desktop_voice_agent.py`，以 WAV 檔案作為音訊來源。
